## Exercise 6 - Feature Engineering

**Anggota Kelompok :**

*1. Kane Matthew Prastersen - 25031554141*

*2. Nayla Salsabila Az Zahra - 25031554268*

# **FEATURE ENGINEERING PART 1**

In [1]:
import warnings
warnings.filterwarnings(action='ignore')

import pandas as pd
import numpy as np
import re

## One Hot Encoding & Bag of Words (BoW)

In [3]:
from sklearn.feature_extraction.text import CountVectorizer

documents = ["ihsg menguat rupiah stabil",
             "pemerintah atur subsidi dan stimulus ekonomi",
             "rupiah melemah ihsg dan stimulus pemerintah"]

vectorizer = CountVectorizer(binary=True)
X = vectorizer.fit_transform(documents)
df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
df

,atur,dan,ekonomi,ihsg,melemah,menguat,pemerintah,rupiah,stabil,stimulus,subsidi
0,0,0,0,1,0,1,0,1,1,0,0
1,1,1,1,0,0,0,1,0,0,1,1
2,0,1,0,1,1,0,1,1,0,1,0


In [4]:
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(documents)
df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
df

,atur,dan,ekonomi,ihsg,melemah,menguat,pemerintah,rupiah,stabil,stimulus,subsidi
0,0,0,0,1,0,1,0,1,1,0,0
1,1,1,1,0,0,0,1,0,0,1,1
2,0,1,0,1,1,0,1,1,0,1,0


## TF-IDF

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(documents)
df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
df

,atur,dan,ekonomi,ihsg,melemah,menguat,pemerintah,rupiah,stabil,stimulus,subsidi
0,0.000000,0.000000,0.000000,0.428046,0.00000,0.562829,0.000000,0.428046,0.562829,0.000000,0.000000
1,0.459548,0.349498,0.459548,0.000000,0.00000,0.000000,0.349498,0.000000,0.000000,0.349498,0.459548
2,0.000000,0.385503,0.000000,0.385503,0.50689,0.000000,0.385503,0.385503,0.000000,0.385503,0.000000


## Applied to the preprocess before
Sumber data : CNN Indonesia
Kategori : Ekonomi (https://www.cnnindonesia.com/ekonomi)

Data yang dipakai untuk feature engineering adalah teks judul berita yang sudah dibersihkan (huruf kecil, karakter yang tidak diperlukan dihapus, dan spasi dirapihkan)

In [10]:
import requests
from bs4 import BeautifulSoup

print("Mulai mengambil data dari CNN Indonesia...")

def scrape_page(url):
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers, timeout=20)
    if response.status_code != 200:
        print(f"Gagal: {url} status {response.status_code}")
        return []
    soup = BeautifulSoup(response.text, "html.parser")
    hasil = []
    for a in soup.select("a[href*='/ekonomi/20']"):
        h = a.find(["h2", "h3"])
        title = h.get_text(" ", strip=True) if h else (a.get("dtr-ttl") or "")
        if len(title) > 15:
            hasil.append(title)
    return hasil

# Scraping langsung (halaman 1 = halaman utama ekonomi, halaman 2+ = indeks ekonomi)
titles_live = []
try:
    titles_live += scrape_page("https://www.cnnindonesia.com/ekonomi")
    for page in range(1, 11):
        titles_live += scrape_page(f"https://www.cnnindonesia.com/ekonomi/indeks/5?page={page}")
except Exception as e:
    print("Scraping langsung tidak berhasil, memakai snapshot saja:", e)

titles = titles_live + titles_snapshot
raw_df = pd.DataFrame({"raw_title": titles}).drop_duplicates().reset_index(drop=True)

print(f"Total judul setelah scraping: {len(raw_df)}")
raw_df.head()

Mulai mengambil data dari CNN Indonesia...
Total judul setelah scraping: 114


,raw_title
0,"Mengupas Tuah Kemenko Baru Prabowo, Bawa Berka..."
1,Rekam Jejak 2 Anggota BPK Pilihan Komisi XI DPR
2,VIDEO: Berkah Karbon dari Belantara Jambi
3,IHSG Diramal Lanjut Ngegas Hari Ini
4,"Bea Masuk AS Tinggi, Indonesia Bidik Pasar Pan..."


In [11]:
# Tahap preprocessing ringan
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df_scraped = raw_df.copy()
df_scraped["prepro"] = df_scraped["raw_title"].apply(preprocess_text)

df_scraped = df_scraped[df_scraped["prepro"].str.len() > 0].reset_index(drop=True)

df_scraped.to_csv("cnn_ekonomi_preprocessed.csv", index=False)

print(f"Dataset preprocessing tersimpan: {len(df_scraped)} baris")
df_scraped.head()

Dataset preprocessing tersimpan: 114 baris


,raw_title,prepro
0,"Mengupas Tuah Kemenko Baru Prabowo, Bawa Berka...",mengupas tuah kemenko baru prabowo bawa berkah...
1,Rekam Jejak 2 Anggota BPK Pilihan Komisi XI DPR,rekam jejak 2 anggota bpk pilihan komisi xi dpr
2,VIDEO: Berkah Karbon dari Belantara Jambi,video berkah karbon dari belantara jambi
3,IHSG Diramal Lanjut Ngegas Hari Ini,ihsg diramal lanjut ngegas hari ini
4,"Bea Masuk AS Tinggi, Indonesia Bidik Pasar Pan...",bea masuk as tinggi indonesia bidik pasar pane...


In [12]:
df = pd.read_csv("cnn_ekonomi_preprocessed.csv")
df = df.dropna(subset=["prepro"]).reset_index(drop=True)
df_pre = df["prepro"]

print("Jumlah dokumen:", len(df_pre))
df_pre.head()

Jumlah dokumen: 114


,prepro
0,mengupas tuah kemenko baru prabowo bawa berkah...
1,rekam jejak 2 anggota bpk pilihan komisi xi dpr
2,video berkah karbon dari belantara jambi
3,ihsg diramal lanjut ngegas hari ini
4,bea masuk as tinggi indonesia bidik pasar pane...


In [13]:
# One Hot Encoding pada berita nyata
onehot_vectorizer = CountVectorizer(binary=True, min_df=2, max_features=2000)
X_onehot = onehot_vectorizer.fit_transform(df_pre)

df_onehot = pd.DataFrame(X_onehot.toarray(),
                         columns=onehot_vectorizer.get_feature_names_out())

print("Ukuran matriks One Hot:", df_onehot.shape)
df_onehot.head()

Ukuran matriks One Hot: (114, 173)


,000,100,13,2026,2027,23,28,abk,ada,agustus,...,umkm,umum,ungkap,us,usai,utang,video,xi,yang,yayasan
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [14]:
# Bag of Words pada berita nyata
bow_vectorizer = CountVectorizer(min_df=2, max_features=2000)
X_bow = bow_vectorizer.fit_transform(df_pre)

df_bow = pd.DataFrame(X_bow.toarray(),
                      columns=bow_vectorizer.get_feature_names_out())

print("Ukuran matriks BoW:", df_bow.shape)
df_bow.head()

Ukuran matriks BoW: (114, 173)


,000,100,13,2026,2027,23,28,abk,ada,agustus,...,umkm,umum,ungkap,us,usai,utang,video,xi,yang,yayasan
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [15]:
# TF-IDF pada berita nyata
tfidf_vectorizer = TfidfVectorizer(min_df=2,
                                   max_features=2000,
                                   sublinear_tf=True)

X_tfidf = tfidf_vectorizer.fit_transform(df_pre)

df_FE = pd.DataFrame(X_tfidf.toarray(),
                     columns=tfidf_vectorizer.get_feature_names_out())

print("Ukuran matriks TF-IDF:", df_FE.shape)
df_FE.head()

Ukuran matriks TF-IDF: (114, 173)


,000,100,13,2026,2027,23,28,abk,ada,agustus,...,umkm,umum,ungkap,us,usai,utang,video,xi,yang,yayasan
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.416667,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.605906,0.000000,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0


In [16]:
# Kata dengan bobot TF-IDF rata-rata tertinggi
df_FE.mean().sort_values(ascending=False).head(10)

,0
di,0.052557
ke,0.038381
ini,0.036900
2026,0.036457
as,0.036324
yang,0.035092
ri,0.034998
indonesia,0.033330
harga,0.031856
jadi,0.027591


## Analisis Feature Engineering 1

Hasil analisis setelah notebook dijalankan.

One Hot Encoding menandai apakah sebuah kata muncul atau tidak dalam suatu judul berita. BoW (Bag of Words) menghitung seberapa sering kata tersebut muncul. TF-IDF memberi nilai lebih tinggi pada kata yang khas pada judul tertentu dan nilai lebih rendah pada kata yang terlalu umum (misalnya "yang", "di", "ri"). Karena judul berita sangat pendek, hampir semua nilai One Hot dan BoW identik (kata jarang muncul lebih dari sekali dalam satu judul), sehingga TF-IDF lebih informatif untuk membedakan topik seperti IHSG/rupiah, subsidi/BSU, dan ekspor/perdagangan.

# **FEATURE ENGINEERING PART 2 (WORD EMBEDDINGS)**

In [17]:
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")

from nltk import word_tokenize
from tqdm import tqdm

sentences = [word_tokenize(str(news).lower())
             for news in tqdm(df_pre, desc="Tokenisasi")]

sentences[:2]

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
Tokenisasi: 100%|██████████| 114/114 [00:00<00:00, 1410.77it/s]


[['mengupas',
  'tuah',
  'kemenko',
  'baru',
  'prabowo',
  'bawa',
  'berkah',
  'atau',
  'justru',
  'bencana'],
 ['rekam', 'jejak', '2', 'anggota', 'bpk', 'pilihan', 'komisi', 'xi', 'dpr']]

## Build Word2Vec Model

In [18]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 39.7 MB/s eta 0:00:00


In [19]:
import gensim.models

model = gensim.models.Word2Vec(sentences,
                               min_count=2,
                               vector_size=100,
                               window=5,
                               sg=0,
                               epochs=20,
                               workers=4,
                               seed=42)

parameter Word2Vec

*   min_count=2: hanya kata yang muncul minimal 2 kali.
*   vector_size=100: setiap kata direpresentasikan dalam 100 dimensi.
*   window=5: konteks kata diambil dari jendela 5 kata.
*   sg=0: menggunakan CBOW
*   epochs=20: model dilatih beberapa putaran agar representasi lebih stabil.
*   seed=42: hasil lebih konsisten saat notebook dijalankan ulang.

In [20]:
model.save('ekonomi_cnn.w2v')

In [21]:
model = gensim.models.Word2Vec.load('ekonomi_cnn.w2v')

In [22]:
w2v = model.wv

print("10 kata teratas:")
print(w2v.index_to_key[:10])

print("\nUkuran vocabulary:", len(w2v))
print("Dimensi vektor:", w2v.vector_size)

10 kata teratas:
['di', 'ke', 'ri', 'as', 'ini', 'harga', 'yang', '2026', 'jadi', 't']

Ukuran vocabulary: 182
Dimensi vektor: 100


In [23]:
# Uji kemiripan konteks kata
kata_uji = "rupiah"

if kata_uji in w2v.key_to_index:
    print(w2v.similar_by_key(kata_uji, topn=10))
else:
    print(f"Kata '{kata_uji}' tidak ada di vocabulary.")

[('negara', 0.2541915774345398), ('surat', 0.25241509079933167), ('antam', 0.25123903155326843), ('impor', 0.24264556169509888), ('makanan', 0.2352585345506668), ('global', 0.22538761794567108), ('airlangga', 0.2245156615972519), ('xi', 0.19431035220623016), ('selasa', 0.1824227124452591), ('tembus', 0.16643324494361877)]


In [24]:
# FE result
def get_document_vector(tokens, model):
    valid_vectors = [model.wv[word] for word in tokens
                     if word in model.wv]

    if not valid_vectors:
        return np.zeros(model.vector_size)

    return np.mean(valid_vectors, axis=0)

X_features_w2v = np.array([get_document_vector(doc, model)
                           for doc in sentences])

df_w2v = pd.DataFrame(X_features_w2v,
                      columns=[f"feature_{i+1}" for i in range(model.vector_size)])

print("Ukuran fitur dokumen Word2Vec:", df_w2v.shape)
df_w2v.head()

Ukuran fitur dokumen Word2Vec: (114, 100)


,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,...,feature_91,feature_92,feature_93,feature_94,feature_95,feature_96,feature_97,feature_98,feature_99,feature_100
0,0.001582,0.000049,-0.002594,-0.000839,-0.004808,-0.001341,-0.001174,-0.000407,-0.002088,0.001202,...,0.001067,0.000666,-0.000340,-0.000218,-0.002580,-0.003588,-0.003269,0.006950,0.000286,-0.000773
1,-0.001240,-0.003455,-0.000863,-0.002783,-0.001893,0.001495,0.002099,-0.000737,-0.004704,0.002376,...,-0.004168,0.001761,0.003328,-0.002450,0.000256,0.002103,0.003593,-0.001039,-0.001385,0.001709
2,0.003753,0.000462,-0.001518,-0.004404,-0.005597,0.001650,0.000305,-0.004964,0.000514,-0.005373,...,-0.005172,-0.002643,0.000411,0.002151,-0.002242,0.001564,-0.002343,0.006648,0.004529,-0.002377
3,0.006543,0.000281,-0.001199,0.006564,0.000742,0.003483,0.001015,-0.005083,-0.000180,0.000517,...,0.004335,0.001400,0.003311,-0.002054,-0.000542,-0.000335,-0.001402,-0.001713,-0.000932,-0.004883
4,0.002373,-0.003122,-0.002584,0.002112,0.000835,-0.001763,-0.002980,0.004507,-0.001686,-0.002829,...,0.002056,-0.002579,-0.001744,0.002193,-0.000975,-0.000467,0.002601,-0.001730,0.003151,-0.001571


## Visualize Word2Vec

In [25]:
!pip install umap-learn

In [26]:
from umap import UMAP
import plotly.express as px

In [27]:
TOP_N = min(100, len(w2v))
top_words = w2v.index_to_key[:TOP_N]
vectors_top = np.array([w2v[word] for word in top_words])

reducer = UMAP(n_neighbors=min(10, TOP_N - 1),
               min_dist=0.7,
               spread=1.5,
               metric="cosine",
               init="random",
               random_state=42)

X_2d = reducer.fit_transform(vectors_top)

df2 = pd.DataFrame(X_2d, columns=["umap1", "umap2"])
df2["word"] = top_words

fig = px.scatter(df2,
                 x="umap1",
                 y="umap2",
                 text="word",
                 hover_name="word")

fig.update_traces(textposition="top center",
                  marker=dict(size=7),
                  textfont=dict(size=9))

fig.update_layout(height=850,
                  title_text="Word2Vec Visualization - Berita Ekonomi CNN Indonesia",
                  template="plotly_white")

fig.show()

## Build FastText Model

In [28]:
model_fasttext = gensim.models.FastText(sentences,
                                        min_count=2,
                                        vector_size=100,
                                        window=5,
                                        min_n=3,
                                        max_n=6,
                                        sg=1,
                                        epochs=20,
                                        workers=4,
                                        seed=42)

parameter FastText

*   min_count=2: buang kata yang terlalu jarang muncul.
*   vector_size=100: tiap kata punya 100 angka fitur.
*   window=5: lihat konteks 5 kata di sekitar.
*   min_n=3, max_n=6: pakai potongan huruf 3-6 karakter.
*   sg=1: metode Skip-Gram, tebak kata sekitar.
*   epochs=20: latih model 20 kali putaran.

In [29]:
model_fasttext.save('ekonomi_cnn.fasttext')

In [30]:
model_fasttext = gensim.models.FastText.load('ekonomi_cnn.fasttext')

In [31]:
fastext = model_fasttext.wv

print("10 kata teratas:")
print(fastext.index_to_key[:10])

print("\nUkuran vocabulary:", len(fastext))
print("Dimensi vektor:", fastext.vector_size)

10 kata teratas:
['di', 'ke', 'ri', 'as', 'ini', 'harga', 'yang', '2026', 'jadi', 't']

Ukuran vocabulary: 182
Dimensi vektor: 100


In [32]:
# Uji kemiripan kata
kata_uji = "ekonomi"

if kata_uji in fastext.key_to_index:
    print(fastext.similar_by_key(kata_uji, topn=10))
else:
    print(f"Kata '{kata_uji}' tidak ada di vocabulary.")

[('dan', 0.9317924380302429), ('buat', 0.9307806491851807), ('danantara', 0.9178575277328491), ('menguat', 0.9150161743164062), ('tinggalkan', 0.914400041103363), ('data', 0.9137352705001831), ('alasan', 0.9124592542648315), ('buatan', 0.9124448895454407), ('kemenkeu', 0.912159264087677), ('kemenko', 0.9111248254776001)]


In [33]:
# FE result
def get_document_vector_fasttext(tokens, model):
    valid_vectors = [model.wv[word] for word in tokens
                     if word in model.wv]

    if not valid_vectors:
        return np.zeros(model.vector_size)

    return np.mean(valid_vectors, axis=0)

X_features_fasttext = np.array([get_document_vector_fasttext(doc, model_fasttext)
                                for doc in sentences])

df_fasttext = pd.DataFrame(X_features_fasttext,
                           columns=[f"feature_{i+1}" for i in range(model_fasttext.vector_size)])

print("Ukuran fitur dokumen FastText:", df_fasttext.shape)
df_fasttext.head()

Ukuran fitur dokumen FastText: (114, 100)


,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,...,feature_91,feature_92,feature_93,feature_94,feature_95,feature_96,feature_97,feature_98,feature_99,feature_100
0,0.001483,-0.000522,-0.000596,-0.001963,0.000961,0.006202,-0.000378,-0.002547,0.002151,0.002109,...,-0.004301,0.003257,0.003187,0.000440,-0.000669,0.002689,-0.000481,-0.001136,0.000924,-0.001591
1,0.001731,0.000006,0.000412,-0.001262,0.001693,0.004677,-0.001172,-0.004864,0.001626,0.001855,...,-0.004527,0.003590,0.004505,-0.000393,-0.000521,0.005101,0.000573,-0.003149,0.001120,-0.001196
2,0.001854,-0.000190,0.000097,-0.001210,0.000966,0.004641,-0.000161,-0.002815,0.001726,0.002128,...,-0.003219,0.002541,0.003591,0.000362,-0.000935,0.003113,0.000515,-0.000574,0.000960,-0.001276
3,0.002135,0.000640,-0.001602,-0.000625,0.001221,0.005120,-0.000825,-0.003190,0.003144,0.003594,...,-0.003317,0.003452,0.005012,0.000042,-0.001280,0.003659,-0.000782,-0.003279,0.000792,-0.001529
4,0.001855,0.000808,-0.000896,-0.001975,0.002477,0.004849,-0.000487,-0.004109,0.001515,0.002395,...,-0.003332,0.002773,0.003296,0.001364,-0.000491,0.002156,-0.000327,-0.001742,0.001345,-0.002269


## Visualize FastText

In [34]:
# dengan vocabulary terbatas
TOP_N = min(100, len(fastext))
top_words = fastext.index_to_key[:TOP_N]
vectors_top = np.array([fastext[word] for word in top_words])

reducer = UMAP(n_neighbors=min(10, TOP_N - 1),
               min_dist=0.7,
               spread=1.5,
               metric="cosine",
               init="random",
               random_state=42)

X_2d = reducer.fit_transform(vectors_top)

df2 = pd.DataFrame(X_2d, columns=["umap1", "umap2"])
df2["word"] = top_words

fig = px.scatter(df2,
                 x="umap1",
                 y="umap2",
                 text="word",
                 hover_name="word")

fig.update_traces(textposition="top center",
                  marker=dict(size=7),
                  textfont=dict(size=9))

fig.update_layout(height=850,
                  title_text="FastText Visualization - Berita Ekonomi CNN Indonesia",
                  template="plotly_white")

fig.show()

## Analisis Feature Engineering Part 2

Hasil analisis setelah notebook dijalankan.

Word2Vec dan FastText sama-sama mengubah kata menjadi vektor. Bedanya, FastText juga memanfaatkan potongan huruf (subword), sehingga kata berimbuhan atau berbentuk mirip tetap bisa direpresentasikan. Karena korpus judul berita ini kecil, hasil kemiripan kata masih terbatas; korpus lebih besar (lebih banyak halaman indeks) akan memberi embedding yang lebih bermakna. UMAP hanya alat bantu visualisasi: titik yang berdekatan menunjukkan vektor yang mirip, tetapi jarak di 2D hanyalah gambaran kasar.